In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Grid
Nx, Ny = 100, 50
Lx, Ly = 1.0, 0.5
dx, dy = Lx/Nx, Ly/Ny

# Constants
gamma = 1.4
CFL = 0.5
t_end = 0.01

# Initial state: supersonic inflow
def initial_conditions():
    rho = np.ones((Nx, Ny)) * 1.0
    u = np.ones((Nx, Ny)) * 2.0
    v = np.zeros((Nx, Ny))
    p = np.ones((Nx, Ny)) * 0.5
    return primitive_to_conserved(rho, u, v, p)

def primitive_to_conserved(rho, u, v, p):
    E = p / (gamma - 1) + 0.5 * rho * (u**2 + v**2)
    return np.stack([rho, rho*u, rho*v, E], axis=0)

def conserved_to_primitive(U):
    rho = U[0]
    u = U[1] / rho
    v = U[2] / rho
    E = U[3]
    p = (gamma - 1) * (E - 0.5 * rho * (u**2 + v**2))
    return rho, u, v, p

# Roe Flux (in x-direction only here)
def roe_flux_x(U_left, U_right):
    # Left and right states
    rho_L, u_L, v_L, p_L = conserved_to_primitive(U_left)
    rho_R, u_R, v_R, p_R = conserved_to_primitive(U_right)

    # Enthalpy
    H_L = (U_left[3] + p_L) / rho_L
    H_R = (U_right[3] + p_R) / rho_R

    # Roe averages
    rL = np.sqrt(rho_L)
    rR = np.sqrt(rho_R)
    r_sum = rL + rR

    u_tilde = (rL * u_L + rR * u_R) / r_sum
    H_tilde = (rL * H_L + rR * H_R) / r_sum
    a_tilde = np.sqrt((gamma - 1) * (H_tilde - 0.5 * u_tilde**2))

    delta_U = U_right - U_left

    # Eigenvalues (wave speeds)
    lam1 = u_tilde - a_tilde
    lam2 = u_tilde
    lam3 = u_tilde + a_tilde

    # Entropy fix
    delta = 0.1 * a_tilde
    lam1 = np.where(np.abs(lam1) < delta, 0.5 * (lam1**2 / delta + delta), np.abs(lam1))
    lam2 = np.where(np.abs(lam2) < delta, 0.5 * (lam2**2 / delta + delta), np.abs(lam2))
    lam3 = np.where(np.abs(lam3) < delta, 0.5 * (lam3**2 / delta + delta), np.abs(lam3))

    # Fluxes left/right
    def flux(U):
        rho, u, v, p = conserved_to_primitive(U)
        F = np.zeros_like(U)
        F[0] = rho * u
        F[1] = rho * u**2 + p
        F[2] = rho * u * v
        F[3] = u * (U[3] + p)
        return F

    F_L = flux(U_left)
    F_R = flux(U_right)

    # Roe dissipation term (simplified)
    # Note: full characteristic decomposition omitted for brevity
    dissipation = 0.5 * np.maximum.reduce([lam1, lam2, lam3]) * (U_right - U_left)

    # Final Roe flux
    return 0.5 * (F_L + F_R) - dissipation

# Time integration
U = initial_conditions()

def compute_dt(U):
    rho, u, v, p = conserved_to_primitive(U)
    a = np.sqrt(gamma * p / rho)
    max_speed = np.max(np.abs(u) + a)
    return CFL * dx / max_speed

# Time loop
t = 0.0
while t < t_end:
    dt = compute_dt(U)
    dt = min(dt, t_end - t)
    
    U_new = U.copy()

    # x-direction flux
    for i in range(1, Nx-1):
        for j in range(1, Ny):
            F_plus = roe_flux_x(U[:, i+1, j], U[:, i, j])
            F_minus = roe_flux_x(U[:, i, j], U[:, i-1, j])
            U_new[:, i, j] -= dt / dx * (F_plus - F_minus)

    # Simple inflow/outflow BCs
    U_new[:, 0, :] = U[:, 1, :]
    U_new[:, -1, :] = U[:, -2, :]
    U_new[:, :, 0] = U[:, :, 1]
    U_new[:, :, -1] = U[:, :, -2]

    U = U_new
    t += dt
    print(f"t = {t:.4f}")

# Plotting density
rho, u, v, p = conserved_to_primitive(U)
plt.imshow(rho.T, origin='lower', extent=[0, Lx, 0, Ly], aspect='auto')
plt.colorbar(label='Density')
plt.title("Density field at final time")
plt.xlabel("x")
plt.ylabel("y")
plt.show()


t = 0.0018
t = 0.0035


KeyboardInterrupt: 

In [ ]:
plt.imshow(u)
plt.colorbar()